# From DICOM RT to a Research-Ready NIfTI Dataset

A complete, runnable walkthrough of [**DicomRTTool**](https://github.com/brianmanderson/Dicom_RT_and_Images_to_Mask)
on a public radiation-therapy cohort: **CT images, RT structures, and RT dose**
in, an analysis-ready NIfTI dataset out.

The data is TCIA's **Pancreatic-CT-CBCT-SEG** collection — chosen because every
patient ships a planning CT, structure sets, *and* an RT dose grid, so the whole
pipeline including dose is exercised on real clinical data.

1. **Download** a subset of the collection from TCIA.
2. **Discover** every series, structure, and dose the walk can find.
3. **Survey** the cohort with a manifest — ROI volumes *and* per-plan Dmax.
4. **Spot outliers** in geometry, contour volume, and dose.
5. **Select & normalize** inconsistent ROI names onto canonical ones.
6. **Choose an output voxel size** and understand the resampling rules.
7. **Preserve clinical metadata** as a JSON sidecar.
8. **Export** images, masks, and dose to NIfTI in one call.
9. **Verify** an exported case end to end.
10. **Grow** the dataset over time without re-identifying anyone.

> Runs top to bottom. See §1 for the download size before you start.

---
## 0 · Setup

`DicomRTTool` does the DICOM→NIfTI work; `tcia_utils` is the client for TCIA's
NBIA API.

In [ ]:
# Run once. Restart the kernel afterwards if pip upgrades an already-loaded package.
%pip install -q --upgrade DicomRTTool tcia_utils SimpleITK pandas matplotlib

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# One base folder for everything this notebook produces.
BASE      = Path("pancreatic_ct_cbct").resolve()
DICOM_DIR = BASE / "dicom"          # raw DICOM downloaded from TCIA
OUT_DIR   = BASE / "nifti"          # the NIfTI dataset we will write
MANIFEST  = BASE / "manifest.csv"   # cohort survey table
for directory in (DICOM_DIR, OUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("Working under:", BASE)

---
## 1 · Download from TCIA

**Collection:** `Pancreatic-CT-CBCT-SEG` — 40 pancreatic cancer patients treated
with SBRT. Each patient has a planning CT, several aligned CBCT/registration
series, structure sets for both, and an **RT dose** grid.

- **Collection page:** <https://www.cancerimagingarchive.net/collection/pancreatic-ct-cbct-seg/>
- **License:** CC BY 4.0 — free to use **with attribution**. Please cite the
  collection if you publish with it.
- **Full size:** ~14 GB for all series.

### What we actually pull

Downloading every series would be ~10 GB for 30 patients, and most of it is
CBCT and registration output this walkthrough does not need. So we take, per
patient, **the planning CT plus every RT structure and RT dose object** — about
110 MB each, so **roughly 3.3 GB for the default 30 patients**.

Lower `N_PATIENTS` if you want a faster first run; nothing downstream depends on
the count.

In [ ]:
from tcia_utils import nbia

COLLECTION = "Pancreatic-CT-CBCT-SEG"

# Series-level metadata for the whole collection (fast — metadata only).
series_df = nbia.getSeries(collection=COLLECTION, format="df")
series_df["ImageCount"] = series_df["ImageCount"].astype(int)

print(f"{len(series_df)} series across {series_df['PatientID'].nunique()} patients")
print(series_df["Modality"].value_counts().to_string())

In [ ]:
N_PATIENTS = 30

def planning_ct(patient_rows: pd.DataFrame) -> pd.DataFrame:
    '''The patient's planning CT — the CT series that isn't a derived product.

    Every patient here carries several CT-modality series: the planning scan
    plus CBCTs and registration output, which this collection names
    ``Aligned ...``. Take the largest series that is *not* one of those; if a
    patient has no named candidate (two of the forty), fall back to the largest
    CT overall.
    '''
    cts = patient_rows[patient_rows["Modality"] == "CT"]
    if cts.empty:
        return cts
    described = cts["SeriesDescription"].fillna("").str.strip()
    candidates = cts[(described != "") & ~described.str.startswith("Aligned")]
    if candidates.empty:
        candidates = cts
    return candidates.nlargest(1, "ImageCount")


patients = sorted(series_df["PatientID"].unique())[:N_PATIENTS]
wanted = pd.concat(
    [
        pd.concat([
            planning_ct(rows),
            rows[rows["Modality"].isin(["RTSTRUCT", "RTDOSE"])],
        ])
        for _, rows in series_df[series_df["PatientID"].isin(patients)].groupby("PatientID")
    ]
)

gb = wanted["FileSize"].astype(float).sum() / 1e9
print(f"{len(patients)} patients -> {len(wanted)} series, about {gb:.1f} GB")
print(wanted["Modality"].value_counts().to_string())

In [ ]:
# Downloads into DICOM_DIR/<SeriesInstanceUID>/*.dcm — a few minutes.
nbia.downloadSeries(wanted, input_type="df", path=str(DICOM_DIR))
print("Done. DICOM under:", DICOM_DIR)

### Alternative — the NBIA Data Retriever

For large pulls TCIA recommends its desktop client: click **Download** on the
collection page to get a `.tcia` manifest, open it in the free **NBIA Data
Retriever**, then point `DICOM_DIR` at the folder it fills. `tcia_utils` can
also consume that manifest directly:

```python
nbia.downloadSeries("Pancreatic-CT-CBCT-SEG.tcia", input_type="manifest", path=str(DICOM_DIR))
```

---
## 2 · Discover — walk the folders

`walk_through_folders` recursively scans the tree, groups files by
`SeriesInstanceUID`, and links each RT structure and dose back to its image
series. They do **not** need to share a folder — here every series landed in its
own UID-named directory, and the walk stitches them back together.

We build the reader with everything it needs up front — the extra DICOM tags to
harvest (§7) and `get_dose_output=True` — so the tree is walked exactly **once**.
Tags are read during the walk, so adding them later would mean walking again.

In [ ]:
from DicomRTTool.ReaderWriter import DicomReaderWriter, ROIAssociationClass

# Non-identifying DICOM tags worth keeping. SITK reads tags by "group|element".
# Values are written to the sidecar verbatim, so never request PatientName or
# PatientID into an anonymized export.
EXTRA_TAGS = {
    "PatientAge":            "0010|1010",
    "PatientSex":            "0010|0040",
    "Manufacturer":          "0008|0070",
    "ManufacturerModelName": "0008|1090",
    "KVP":                   "0018|0060",
    "SliceThickness":        "0018|0050",
}

reader = DicomReaderWriter(
    image_sitk_string_keys=EXTRA_TAGS,
    get_dose_output=True,        # load and export the dose grids
    require_all_contours=False,  # keep series carrying only some of the ROIs
    verbose=False,
)
reader.walk_through_folders(str(DICOM_DIR))

print(f"{len(reader.series_instances_dictionary)} image series | "
      f"{len(reader.rt_dictionary)} structure sets | "
      f"{len(reader.rd_dictionary)} dose series | "
      f"{len(reader.rp_dictionary)} plans")

In [ ]:
# Every ROI name present across everything we downloaded.
rois = reader.return_rois(print_rois=True)

Note the **plan count is zero**. This collection ships RT dose *without* the RT
plan that produced it — common in public data. It matters below: a dose is
normally labelled by its plan's `RTPlanName`, so with no plan available
DicomRTTool falls back to the dose's own series description.

---
## 3 · Survey — write a metadata manifest

`create_manifest` writes **one row per series**, and there are two kinds, told
apart by the `modality` column:

| Row | `modality` | Value columns |
|-----|-----------|---------------|
| Image series | `CT` | one `<roi> cc` per ROI — the mask volume |
| Dose series | `RTDOSE` | one `<plan> cGy` per plan — that plan's **Dmax** |

A dose row is keyed by the **dose series' own** identifiers, so it shares the
patient and study of the images it belongs to but has its own series hash, and
its spacing is the dose grid's own. Each row fills only its own kind of column.

Dmax is measured on the **native** dose grid, before any resampling — pushing a
dose onto a coarser grid interpolates its peak away, and the survey should
report the real maximum.

In [ ]:
reader.create_manifest(str(MANIFEST))

manifest = pd.read_csv(MANIFEST)
print("Shape:", manifest.shape)
print(manifest["modality"].value_counts().to_string())
manifest.head()

In [ ]:
# Split the two row kinds apart.
images = manifest[manifest["modality"] != "RTDOSE"]
doses  = manifest[manifest["modality"] == "RTDOSE"]

cc_cols  = [c for c in manifest.columns if c.endswith(" cc")]
cgy_cols = [c for c in manifest.columns if c.endswith(" cGy")]

print(f"{len(images)} image rows, {len(doses)} dose rows")
print("ROI volume columns :", cc_cols)
print("Plan dose columns  :", cgy_cols)

# One Dmax per dose row, whichever plan column it landed in.
dmax = doses[cgy_cols].max(axis=1)
print("\nDmax (cGy) across the cohort:")
print(dmax.describe().round(1).to_string())

---
## 4 · Spot the outliers

The manifest is your first QC pass. Four things to look for:

- **Geometry** — an odd `spacing_z` (a 5 mm slice in a 3 mm cohort).
- **Contour volume** — an ROI far from the cohort norm (partial or mis-drawn).
- **Missing structures** — a blank `cc` cell means that ROI isn't on the series.
- **Dose** — a Dmax well off the cohort norm usually means a different
  prescription, or a dose grid that doesn't belong to the plan you think it does.

In [ ]:
spacing_cols = [c for c in manifest.columns if c.startswith("spacing_")]
print("Image geometry:")
print(images[spacing_cols].describe().round(3).to_string())
print("\nDose grid geometry:")
print(doses[spacing_cols].describe().round(3).to_string())

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 3.4))

axes[0].hist(images["spacing_z"].dropna(), bins=20, color="#007DBA")
axes[0].set_title("image spacing_z (mm)")
axes[0].set_xlabel("slice thickness"); axes[0].set_ylabel("series")

vol_col = next((c for c in cc_cols if images[c].notna().sum() > 1), None)
if vol_col:
    axes[1].hist(images[vol_col].dropna(), bins=20, color="#1C7293")
    axes[1].set_title(vol_col)
    axes[1].set_xlabel("volume (cc)"); axes[1].set_ylabel("series")

axes[2].hist(dmax.dropna(), bins=20, color="#B03A2E")
axes[2].set_title("Dmax (cGy)")
axes[2].set_xlabel("maximum dose"); axes[2].set_ylabel("dose series")

plt.tight_layout(); plt.show()

In [ ]:
def flag_outliers(series: pd.Series, k: float = 1.5) -> pd.Series:
    '''Boolean mask of IQR outliers: beyond k*IQR below Q1 or above Q3.'''
    values = series.dropna()
    if len(values) < 4:
        return pd.Series(False, index=series.index)
    q1, q3 = values.quantile(0.25), values.quantile(0.75)
    iqr = q3 - q1
    return (series < q1 - k * iqr) | (series > q3 + k * iqr)


report = manifest[["patient_hash", "series_hash", "modality", "spacing_z"]].copy()
report["spacing_z_outlier"] = flag_outliers(manifest["spacing_z"])
for column in cc_cols + cgy_cols:
    report[f"{column} outlier"] = flag_outliers(manifest[column])

flagged = report[report.filter(like="outlier").any(axis=1)]
print(f"{len(flagged)} of {len(manifest)} series flagged for review")
flagged.head(20)

---
## 5 · Select ROIs & normalize their names

Real ROI names are inconsistent, and this collection is a good example: the same
anatomy is contoured on the planning CT and on the CBCT under different names
(`Bowel_sm_planCT` vs `Bowel_sm_CBCT`). `ROIAssociationClass` maps any number of
aliases onto one canonical name.

**Adjust the lists below to whatever `return_rois()` printed above** — matching
is case-insensitive, and an alias that matches nothing is simply ignored.

In [ ]:
CONTOUR_NAMES = ["bowel", "stomach_duo", "lung_l", "lung_r"]

reader.set_contour_names_and_associations(
    contour_names=CONTOUR_NAMES,
    associations=[
        ROIAssociationClass("bowel",       ["bowel_sm_planct", "bowel_sm_cbct", "bowel_sm", "bowel"]),
        ROIAssociationClass("stomach_duo", ["stomach_duo_planct", "stomach_duo_cbct", "stomach_duo"]),
        ROIAssociationClass("lung_l",      ["lung_l", "lung-left", "left lung"]),
        ROIAssociationClass("lung_r",      ["lung_r", "lung-right", "right lung"]),
    ],
)

print("Series carrying the selected ROIs:", len(reader.indexes_with_contours))

---
## 6 · Set the output voxel size

Native spacing varies across the cohort (§4). Pick **one** target so every case
lands on the same grid:

- **Isotropic** `(1.0, 1.0, 1.0)` — uniform and model-friendly, larger arrays.
- **Native-like** `(1.0, 1.0, 3.0)` — smaller, closer to the acquired geometry.

`write_to_folder` applies the right rule per output automatically: **linear**
interpolation for the image and dose, **nearest-neighbour** for masks — a label
must never be blended. The dose is resampled straight onto the resampled image
grid, so image, masks, and dose all come out the same size and geometry.

In [ ]:
OUTPUT_SPACING = (1.0, 1.0, 3.0)   # (x, y, z) in mm — None keeps native spacing
print("Target voxel size (mm):", OUTPUT_SPACING)

---
## 7 · Preserve clinical metadata

Pixels alone aren't research-ready. The `EXTRA_TAGS` requested back in §2 are
written into a **`metadata.json`** beside every exported series.

That sidecar is a grouped, versioned document (`"schema_version": 2`). DICOM
features are organized by category — `image`, `structures`, `doses`, `plans` —
and your requested tags land inside the owning category's `tags` sub-dict
(image tags under `image["tags"]`). Categories with no matching files are
omitted, so an image-only series still parses cleanly via
`meta.get("doses", [])`.

Each dose entry also carries its `plan_name` and `dose_max_cgy`, so a case
folder describes its own dose without needing the cohort manifest.

Pass `metadata_style="flat"` for the historical `{name: value}` dict.

---
## 8 · Export to NIfTI — one call

`write_to_folder` writes every selected series to a tidy per-case tree:
resampled image, one mask per ROI, the dose, and the metadata sidecar — plus a
cohort `manifest.csv` and, because we anonymize, an `anonymization_key.json`.

Hashing is deterministic: the same `salt` always produces the same hashes, which
is what makes the dataset growable (§10).

In [ ]:
reader.write_to_folder(
    str(OUT_DIR),
    output_spacing=OUTPUT_SPACING,
    anonymize=True,
    salt="Pancreatic-CT-CBCT-SEG",
    metadata_style="grouped",
)
print("Exported to:", OUT_DIR)

In [ ]:
def show_tree(root: Path, max_files: int = 6, prefix: str = "") -> None:
    entries = sorted(root.iterdir())
    for file in [e for e in entries if e.is_file()][:max_files]:
        print(prefix + file.name)
    for directory in [e for e in entries if e.is_dir()]:
        print(prefix + directory.name + "/")
        show_tree(directory, max_files, prefix + "    ")


show_tree(OUT_DIR)

---
## 9 · Verify an exported case

Load one exported case back, confirm the image, masks, and dose really do share
a geometry, and read the sidecar.

In [ ]:
import json

import SimpleITK as sitk

cases = sorted(p.parent for p in OUT_DIR.rglob("image.nii.gz"))
assert cases, "No exported images found — re-run section 8."
case = cases[0]
print("Inspecting:", case.relative_to(OUT_DIR))

image = sitk.ReadImage(str(case / "image.nii.gz"))
print("Image size   :", image.GetSize())
print("Image spacing:", tuple(round(s, 3) for s in image.GetSpacing()))

mask_files = sorted((case / "masks").glob("*.nii.gz")) if (case / "masks").exists() else []
dose_files = sorted((case / "doses").glob("*.nii.gz")) if (case / "doses").exists() else []
print("Masks        :", [m.name.replace(".nii.gz", "") for m in mask_files])
print("Doses        :", [d.name.replace(".nii.gz", "") for d in dose_files])

for path in mask_files + dose_files:
    other = sitk.ReadImage(str(path))
    assert other.GetSize() == image.GetSize(), f"{path.name} size differs"
    assert other.GetSpacing() == image.GetSpacing(), f"{path.name} spacing differs"
print("\nAll outputs share the image geometry.")

In [ ]:
meta = json.loads((case / "metadata.json").read_text())
print("schema_version:", meta["schema_version"], "| anonymized:", meta["anonymized"])

image_block = meta["image"]                     # always present
print("image.export :", image_block.get("export"))
print("image.tags   :", image_block.get("tags", {}))   # <- EXTRA_TAGS land here

for structure in meta.get("structures", []):
    for roi in structure["rois"]:
        if "exported_file" in roi:              # only exported ROIs carry these
            canonical = roi.get("canonical_name", roi["name"])
            print(f"  ROI {roi['name']:<22} -> {canonical:<12} "
                  f"{roi['volume_cc']:9.1f} cc  ->  {roi['exported_file']}")

for dose in meta.get("doses", []):
    print(f"  dose plan {dose['plan_name']!r}: Dmax {dose.get('dose_max_cgy')} cGy "
          f"({dose['dose_units']}, {dose['dose_summation_type']})")
print("dose_file:", meta.get("dose_file"))

In [ ]:
# Image, a mask outline, and the dose colourwash on the slice of peak dose.
array = sitk.GetArrayFromImage(image)                       # (z, y, x)

if dose_files:
    dose = sitk.GetArrayFromImage(sitk.ReadImage(str(dose_files[0])))
    z = int(np.unravel_index(np.argmax(dose), dose.shape)[0])
else:
    dose, z = None, array.shape[0] // 2

plt.figure(figsize=(6, 6))
plt.imshow(np.clip(array[z], -200, 300), cmap="gray")

if dose is not None:
    # Show the top half of the dose range only, so low-dose spill stays readable.
    plt.imshow(np.ma.masked_where(dose[z] < dose.max() * 0.5, dose[z]),
               cmap="jet", alpha=0.45)
    plt.colorbar(label="dose (Gy)", fraction=0.046)

for mask_path in mask_files:
    mask = sitk.GetArrayFromImage(sitk.ReadImage(str(mask_path)))
    if mask[z].any():
        plt.contour(mask[z], levels=[0.5], colors="w", linewidths=1.2)

plt.title(f"{case.name} — z={z}")
plt.axis("off"); plt.show()

---
## 10 · Grow the dataset over time

`anonymization_key.json` maps each hash back to its MRN. **It is
re-identification data** — keep it access-controlled, never commit it, and never
ship it with the imaging. The repo's `.gitignore` already excludes it.

Because hashing is deterministic (same `salt` → same hash), you can re-pull the
same patients later and fold in new imaging or follow-up without re-identifying
anything. `create_manifest` and `write_to_folder` are both **incremental**:
point them at an existing manifest and they *upsert* by `series_hash` — existing
rows updated, new series appended — so one manifest can grow across many walks.

```python
# Later, after downloading more patients into DICOM_DIR:
reader.reset()
reader.walk_through_folders(str(DICOM_DIR))
reader.create_manifest(str(MANIFEST))    # updates the same file in place
```

### Where to go next

- **`README`** — the full API: loading a single series into NumPy/SimpleITK,
  writing model predictions back out as an RT structure, resampling helpers,
  and performance tuning.
- **Tool:** <https://github.com/brianmanderson/Dicom_RT_and_Images_to_Mask> · `pip install DicomRTTool`